In [0]:
#Load the existing tables
from pyspark.sql import functions as F
from pyspark.sql.window import Window

customer_df = spark.table("customer_master")
order_df = spark.table("order_master")
order_line_df = spark.table("order_line")
support_df = spark.table("support_case")

print("All tables loaded successfully!")

In [0]:
#Calculate order-level spending
order_amount_df = (
    order_line_df
    .groupBy("order_id")
    .agg(
        F.sum("line_total").alias("order_amount")
    )
)

display(order_amount_df)

In [0]:
#Calculate customer purchase details
delivered_orders_df = (
    order_df
    .filter(F.col("order_status") == "DELIVERED")
    .join(order_amount_df, "order_id", "left")
)

customer_purchase_df = (
    delivered_orders_df
    .groupBy("customer_id")
    .agg(
        F.countDistinct("order_id").alias("frequency"),
        F.sum(F.coalesce(F.col("order_amount"), F.lit(0))).alias("monetary"),
        F.max("order_date").alias("last_purchase_date")
    )
)

display(customer_purchase_df)

In [0]:
#Calculate Recency
analysis_date = F.lit("2026-10-30").cast("date")

rfm_df = (
    customer_df
    .join(customer_purchase_df, "customer_id", "left")
    .withColumn(
        "frequency",
        F.coalesce(F.col("frequency"), F.lit(0))
    )
    .withColumn(
        "monetary",
        F.coalesce(F.col("monetary"), F.lit(0))
    )
    .withColumn(
        "recency",
        F.datediff(analysis_date, F.col("last_purchase_date"))
    )
)

display(rfm_df)

In [0]:
#Save the RFM table
rfm_df.write.format("delta").mode("overwrite").saveAsTable("customer_rfm")

print("Customer RFM table created successfully!")

In [0]:
#Validate the result
print("Total customers:", rfm_df.count())

rfm_df.select(
    F.sum(F.when(F.col("frequency") == 0, 1).otherwise(0)).alias("customers_without_purchases"),
    F.sum(F.when(F.col("frequency") > 0, 1).otherwise(0)).alias("customers_with_purchases")
).show()

In [0]:
spark.table("customer_rfm").display()